## testing of some model and ewatercycle related settings

In [1]:
import sys
from pathlib import Path

import ewatercycle.forcing
import ewatercycle.models
import ewatercycle.observation.grdc
import ewatercycle.parameter_sets
from ewatercycle.container import ContainerImage
from rich import print

PROJECT_ROOT = Path().resolve().parents[2]  # pas aan als notebook dieper/dichter zit
sys.path.append(str(PROJECT_ROOT))

from datetime import datetime

import pandas as pd
from tqdm.notebook import tqdm

from src.constants import STATIONS_PCR
from src.paths import (
    FORCING_PCRGLOB,
    INI_FILES,
    LOAD_PCR,
    OUTPUT_PCRGLOB,
    PCR_GLOBAL_PARAMS,
    PCR_TAIL,
)

/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498


In [2]:
pcr_glob_directory = PCR_GLOBAL_PARAMS  # GlobalOption uit .ini
prepared_pcr_forcing = FORCING_PCRGLOB / LOAD_PCR / PCR_TAIL


parameter_set_full = ewatercycle.parameter_sets.ParameterSet(
    name="custom_parameter_set",
    directory=pcr_glob_directory,
    config=INI_FILES / "test_comp_speed_full.ini",
    target_model="pcrglobwb",
    supported_model_versions={"17feb"},
)

parameter_set_basin = ewatercycle.parameter_sets.ParameterSet(
    name="custom_parameter_set",
    directory=pcr_glob_directory,
    config=INI_FILES / "test_comp_speed_basin.ini",
    target_model="pcrglobwb",
    supported_model_versions={"17feb"},
)

forcing = ewatercycle.forcing.sources["PCRGlobWBForcing"].load(
    directory=prepared_pcr_forcing,
)

In [3]:
my_image = ContainerImage("/home/avandervee3/ewatercycle_pcr_17feb.sif")
my_image.version

'17feb'

In [4]:
reference = ewatercycle.models.PCRGlobWB(
    parameter_set=parameter_set_full, forcing=forcing, bmi_image=my_image
)

reference_2x = ewatercycle.models.PCRGlobWB(
    parameter_set=parameter_set_basin, forcing=forcing, bmi_image=my_image
)

In [5]:
experiment_start_date = "1950-01-01T00:00:00Z"
experiment_end_date = "1950-06-30T00:00:00Z"

In [6]:
reference_config, reference_dir = reference.setup(
    cfg_dir=OUTPUT_PCRGLOB / "full",
    start_time=experiment_start_date,
    end_time=experiment_end_date,
    max_spinups_in_years=0,
)
reference_config, reference_dir

reference_2x_config, reference_2x_dir = reference_2x.setup(
    cfg_dir=OUTPUT_PCRGLOB / "slice",
    start_time=experiment_start_date,
    end_time=experiment_end_date,
    max_spinups_in_years=0,
)
reference_2x_config, reference_2x_dir

('/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/work_in_progress/outputs/model_runs/pcr-globwb/slice/pcrglobwb_ewatercycle.ini',
 '/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/work_in_progress/outputs/model_runs/pcr-globwb/slice')

In [7]:
print(reference.parameters)

refence_para = reference.parameters

# Convert ISO 8601 strings to datetime objects
start_time = datetime.strptime(experiment_start_date, "%Y-%m-%dT%H:%M:%SZ")
end_time = datetime.strptime(experiment_end_date, "%Y-%m-%dT%H:%M:%SZ")

# Calculate the number of days for the progression bar
delta = end_time - start_time
number_of_days = delta.days
print(f"Number of days to model: {number_of_days}")

dict_items([('start_time', '1950-01-01T00:00:00Z'), ('end_time', '1950-06-30T00:00:00Z'), ('routing_method', 
'accuTravelTime'), ('max_spinups_in_years', '0')])

Number of days to model: 180

In [8]:
reference.initialize(reference_config)
reference_2x.initialize(reference_2x_config)

In [9]:
time = pd.date_range(reference.start_time_as_isostr, reference.end_time_as_isostr)
stations_timeseries = pd.DataFrame(
    index=pd.Index(time, name="time"), columns=["Chatly", "Kerki", "Tyumen", "Kazalinsk"]
)
stations_timeseries.head()


stations_discharge_reference = stations_timeseries.copy()
stations_discharge_reference_2x = stations_timeseries.copy()

stations_channelstorage_reference = stations_timeseries.copy()
stations_channelstorage_reference_2x = stations_timeseries.copy()

In [10]:
for _ in tqdm(range(number_of_days), desc="Running model"):
    reference.update()

    for station_name, coords in STATIONS_PCR.items():
        discharge_getvalue = reference.get_value_at_coords(
            "discharge",
            lat=[coords["lat"]],
            lon=[coords["lon"]],
        )

        stations_discharge_reference.loc[time, station_name] = discharge_getvalue[0]


print("Model run finished!")

Running model:   0%|          | 0/180 [00:00<?, ?it/s]

Model run finished!

In [11]:
for _ in tqdm(range(number_of_days), desc="Running model"):
    reference_2x.update()

    for station_name, coords in STATIONS_PCR.items():
        discharge_getvalue = reference_2x.get_value_at_coords(
            "discharge",
            lat=[coords["lat"]],
            lon=[coords["lon"]],
        )

        stations_discharge_reference_2x.loc[time, station_name] = discharge_getvalue[0]

Running model:   0%|          | 0/180 [00:00<?, ?it/s]

In [12]:
reference_2x.finalize()
reference.finalize()